In [2]:
pip install numpy pandas torch torchvision pillow scikit-learn matplotlib opencv-python grad-cam timm --break-system-packages

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import cv2
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## Data Pipeline

In [4]:
# reads classification dataset
df = pd.read_csv("GALAXY_DATASET/final_15000.csv")
df.head()

,dr7objid,asset_id,t01_smooth_or_features_a01_smooth_debiased,t01_smooth_or_features_a02_features_or_disk_debiased,t02_edgeon_a04_yes_debiased,t03_bar_a06_bar_debiased,t04_spiral_a08_spiral_debiased,t08_odd_feature_a24_merger_debiased,t08_odd_feature_a22_irregular_debiased
0,587722981742018657,15,0.003,0.955,0.000000,0.000000,0.733360,0.003648,0.526693
1,587722981744050315,35,0.000,0.981,0.000289,0.402361,0.998695,0.000000,0.045512
2,587722981746016481,63,0.020,0.973,0.000000,0.216366,0.995563,0.000000,0.437910
3,587722981747654761,81,0.766,0.757,0.108623,0.381713,0.926622,0.000000,0.889990
4,587722981747982465,89,0.154,0.846,0.003660,0.885386,0.839584,0.211087,0.340519


In [5]:
img_folder = "GALAXY_DATASET/Images Differing sizes/224"

# 10 images are missing from the folder, drop those rows
valid = set(int(f.replace(".jpg", "")) for f in os.listdir(img_folder) if f.endswith(".jpg"))
df = df[df["asset_id"].isin(valid)].reset_index(drop=True)
print(len(df))  # should be 14990

607


In [6]:
# label generator function, based off of original tree design, but modified slightly for simplicity
def make_label(row, thresh=0.6):

    # only 4 labels here
    smooth  = row["t01_smooth_or_features_a01_smooth_debiased"]
    bar     = row["t03_bar_a06_bar_debiased"]
    spiral  = row["t04_spiral_a08_spiral_debiased"]
    edgeon  = row["t02_edgeon_a04_yes_debiased"]

    if edgeon > thresh:
        return "edge_on"
    elif bar > thresh:
        return "barred_spiral"
    elif spiral > thresh:
        return "spiral"
    elif smooth > thresh:
        return "smooth"
    else:
        return "featured"

df["label"] = df.apply(make_label, axis=1)
df = df.dropna(subset=["label"]).reset_index(drop=True)
print(df["label"].value_counts())

label
spiral           210
smooth           134
barred_spiral    106
featured         102
edge_on           55
Name: count, dtype: int64


In [7]:
# label system for classification
classes = sorted(df["label"].dropna().unique())

print(classes)
print(f"Number of classes: {len(classes)}")

class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

df["label_idx"] = df["label"].map(class_to_idx)

['barred_spiral', 'edge_on', 'featured', 'smooth', 'spiral']
Number of classes: 5


In [8]:
# defines a loader for the galaxy dataset, returns img and labels
class GalaxyDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        path = os.path.join(self.img_dir, f"{int(row['asset_id'])}.jpg")
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(row["label_idx"])

In [9]:
# sets up the data pipeline
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df["label"])
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

# resizes and normalises
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(360),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])
val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# data loaders for batch separation
train_loader = DataLoader(GalaxyDataset(train_df, img_folder, train_tfms), batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(GalaxyDataset(val_df,   img_folder, val_tfms),   batch_size=32, shuffle=False, num_workers=0)
test_loader  = DataLoader(GalaxyDataset(test_df,  img_folder, val_tfms),   batch_size=32, shuffle=False, num_workers=0)

print(f"train: {len(train_loader.dataset)}  val: {len(val_loader.dataset)}  test: {len(test_loader.dataset)}")

train: 424  val: 91  test: 92


In [10]:
# runs each epoch given a model, loader, criterion, and optimizer, returns total_loss and correct
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss, correct = 0.0, 0
    with torch.set_grad_enabled(training):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            loss = criterion(out, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(imgs)
            correct += (out.argmax(1) == labels).sum().item()

    n = len(loader.dataset)
    return total_loss / n, correct / n

## Model

In [11]:
# used for simplifying model parameters/changes
import timm

accuracies = []
# tested accuracies of 3 models, repeated training once for each
for model_name in ["resnet50", "efficientnet_b3", "convnext_small"]:
    
    model = timm.create_model(model_name, pretrained=True, num_classes=len(classes))
    
    for name, param in model.named_parameters():
        if not any(k in name for k in ["fc", "head", "classifier"]):
            param.requires_grad = False
        
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.get_classifier().parameters(), lr=1e-3)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    EPOCHS_P1 = 10
    best_val_acc = 0.0

    # first set of epochs
    for epoch in range(EPOCHS_P1):
        tl, ta = run_epoch(model, train_loader, criterion, optimizer)
        vl, va = run_epoch(model, val_loader,   criterion)
    
        history["train_loss"].append(tl)
        history["val_loss"].append(vl)
        history["train_acc"].append(ta)
        history["val_acc"].append(va)
    
        if va > best_val_acc:
            best_val_acc = va
            torch.save(model.state_dict(), "best_model.pth")
    
        print(f"epoch {epoch+1}/{EPOCHS_P1}  train_loss={tl:.4f}  val_loss={vl:.4f}  train_acc={ta:.4f}  val_acc={va:.4f}")


    for param in model.parameters():
        param.requires_grad = True

    # new optimizer defined
    optimizer = torch.optim.Adam([
        {"params": model.get_classifier().parameters(),  "lr": 1e-3},
        {"params": [p for name, p in model.named_parameters() if not any(k in name for k in ["fc", "head", "classifier"])], "lr": 1e-4}
    ])

    # seconds set of epochs, final training
    EPOCHS_P2 = 15
    p1_done = len(history["val_acc"])
    
    for epoch in range(EPOCHS_P2):
        tl, ta = run_epoch(model, train_loader, criterion, optimizer)
        vl, va = run_epoch(model, val_loader,   criterion)
    
        history["train_loss"].append(tl)
        history["val_loss"].append(vl)
        history["train_acc"].append(ta)
        history["val_acc"].append(va)
    
        if va > best_val_acc:
            best_val_acc = va
            torch.save(model.state_dict(), "best_model.pth")
    
        print(f"epoch {p1_done+epoch+1}  train_loss={tl:.4f}  val_loss={vl:.4f}  train_acc={ta:.4f}  val_acc={va:.4f}")
    
    print(f"\nbest val acc: {best_val_acc:.4f}")
    _, test_acc = run_epoch(model, test_loader, criterion)
    print(f"{model_name} test acc: {test_acc:.4f}")
    
    # appends accuracies for analysis later
    accuracies.append([model_name, f"{test_acc:.4f}"])

epoch 1/10  train_loss=1.5492  val_loss=1.5385  train_acc=0.3278  val_acc=0.3407
epoch 2/10  train_loss=1.5286  val_loss=1.5418  train_acc=0.3538  val_acc=0.3407
epoch 3/10  train_loss=1.5000  val_loss=1.5224  train_acc=0.3467  val_acc=0.3407
epoch 4/10  train_loss=1.5010  val_loss=1.5215  train_acc=0.3467  val_acc=0.3626
epoch 5/10  train_loss=1.4861  val_loss=1.5138  train_acc=0.3467  val_acc=0.3516
epoch 6/10  train_loss=1.4774  val_loss=1.5104  train_acc=0.3585  val_acc=0.3516
epoch 7/10  train_loss=1.4815  val_loss=1.4945  train_acc=0.3679  val_acc=0.3956
epoch 8/10  train_loss=1.4663  val_loss=1.4882  train_acc=0.3585  val_acc=0.3736
epoch 9/10  train_loss=1.4536  val_loss=1.4759  train_acc=0.3632  val_acc=0.4066
epoch 10/10  train_loss=1.4412  val_loss=1.4682  train_acc=0.3821  val_acc=0.4176
epoch 11  train_loss=1.4389  val_loss=1.4808  train_acc=0.3774  val_acc=0.3736
epoch 12  train_loss=1.4087  val_loss=1.4556  train_acc=0.3868  val_acc=0.3956
epoch 13  train_loss=1.3616  va

KeyboardInterrupt: 

In [ ]:
# creates chart for each model and their accuracies
print(accuracies)
plt.figure()
values = []

for i in accuracies:
    values.append(float(i[1]))

# plotting
plt.bar(["ResNet50", "EfficientNetB3", "ConvNeXt"], values, width=0.4, color=["steelblue", "coral", "mediumseagreen"])
plt.ylabel("Test Accuracy")
plt.title("Test Accuracy per Model")
plt.savefig("test_accuracy.png")
plt.show()
plt.close()
